# Protein-only RIN: NAMD PSF/DCD → centrality → hubs → summary

Batch workflow for independent NAMD trajectories. The notebook reads full PSF/DCD systems, selects only `protein`, optionally writes a protein-only DCD cache through MDAnalysis, constructs residue-level protein–protein consensus graphs, performs QC, computes centralities and hubs, and creates per-system and aggregate CSV outputs.

No VMD, lipids, glycans, ESM, structural ML features, communities, allosteric paths, or cross-condition statistics are included.

## Input contract

Create `systems.csv` next to the notebook or change `SYSTEMS_CSV` below. Required columns: `system_id`, `batch`, `length_ns`, `psf_path`, `dcd_path`. Optional columns are retained in results.

Example:
```csv
system_id,batch,replicate,length_ns,psf_path,dcd_path,notes
rep100_01,1,1,100,/data/rep100_01/system.psf,/data/rep100_01/traj.dcd,replicate_1
rep100_02,1,2,100,/data/rep100_02/system.psf,/data/rep100_02/traj.dcd,replicate_2
rep100_03,1,3,100,/data/rep100_03/system.psf,/data/rep100_03/traj.dcd,replicate_3
sys30_01,2,1,30,/data/sys30_01/system.psf,/data/sys30_01/traj.dcd,independent_system
```

Run batch 1 first (`RUN_BATCHES=[1]`), then change to `RUN_BATCHES=[2]`. The notebook uses `results/status.csv` and will skip already completed systems unless `FORCE_RERUN=True`.

In [ ]:
import sys, subprocess, importlib

REQUIRED = {
    'MDAnalysis': 'MDAnalysis>=2.7',
    'networkx': 'networkx>=3.0',
    'pandas': 'pandas>=2.0',
    'numpy': 'numpy>=1.24',
    'tqdm': 'tqdm>=4.0',
}

for module, package in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import gc
import json
import math
import time
import traceback
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import MDAnalysis as mda
import networkx as nx
import numpy as np
import pandas as pd
from MDAnalysis.lib.distances import capped_distance, calc_angles
from tqdm.auto import tqdm

print('MDAnalysis', mda.__version__)
print('networkx', nx.__version__)


In [ ]:
# =========================== USER CONFIGURATION ===========================
SYSTEMS_CSV = Path('systems.csv')
RESULTS_DIR = Path('rin_centrality_results')
PER_SYSTEM_DIR = RESULTS_DIR / 'per_system'
CACHE_DIR = RESULTS_DIR / 'protein_only_cache'

# First run: [1]; second run: [2]; all systems: [1, 2].
RUN_BATCHES = [1]
FORCE_RERUN = False

# Writing a protein-only DCD via MDAnalysis is optional but recommended when
# repeated analyses are planned. The input PSF remains full-system; cached DCD
# is used only for I/O reduction after a protein-only in-memory Universe is made.
WRITE_PROTEIN_ONLY_DCD_CACHE = False

# For the first rapid pass. Increase/decrease only after checking actual frame spacing.
FRAME_STRIDE = 10
MIN_CONTACT_FREQUENCY = 0.30

# Hub definition: nodes at or above this within-graph quantile.
HUB_QUANTILE = 0.90
MIN_HUBS_IF_NONEMPTY = 5

# Contact definitions (Angstrom, degrees).
HBOND_DISTANCE_A = 3.5
HBOND_ANGLE_DEG = 120.0
VDW_MIN_A = 3.0
VDW_MAX_A = 4.5
HYDROPHOBIC_CUTOFF_A = 5.0
AROMATIC_PI_CUTOFF_A = 7.0
AROMATIC_PARALLEL_MAX_DEG = 30.0
AROMATIC_T_MIN_DEG = 60.0
AROMATIC_T_MAX_DEG = 120.0
CATION_PI_CUTOFF_A = 6.0
CATION_PI_NORMAL_MAX_DEG = 60.0
IONIC_CUTOFF_A = 6.0
SALTBRIDGE_CUTOFF_A = 4.0
BACKBONE_CA_MAX_A = 4.5

# Avoid trivial sequential contacts for non-backbone contact types.
SEQUENCE_SEPARATION_MIN = 2

# For expensive shortest-path centralities, use the largest connected component.
CENTRALITY_ON_GIANT_COMPONENT = True

for directory in [RESULTS_DIR, PER_SYSTEM_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_CONFIG = {
    'run_batches': RUN_BATCHES,
    'frame_stride': FRAME_STRIDE,
    'min_contact_frequency': MIN_CONTACT_FREQUENCY,
    'hub_quantile': HUB_QUANTILE,
    'sequence_separation_min': SEQUENCE_SEPARATION_MIN,
    'centrality_on_giant_component': CENTRALITY_ON_GIANT_COMPONENT,
    'edge_types': ['backbone', 'hbond', 'disulfide', 'vdw', 'hydrophobic', 'aromaticpi', 'cationpi', 'ionic', 'saltbridge'],
}
(RESULTS_DIR / 'run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print(json.dumps(RUN_CONFIG, indent=2))


In [ ]:
REQUIRED_COLUMNS = {'system_id', 'batch', 'length_ns', 'psf_path', 'dcd_path'}

if not SYSTEMS_CSV.exists():
    template = pd.DataFrame([
        {'system_id': 'rep100_01', 'batch': 1, 'replicate': 1, 'length_ns': 100, 'psf_path': '/data/rep100_01/system.psf', 'dcd_path': '/data/rep100_01/traj.dcd', 'notes': 'replace_paths'},
        {'system_id': 'rep100_02', 'batch': 1, 'replicate': 2, 'length_ns': 100, 'psf_path': '/data/rep100_02/system.psf', 'dcd_path': '/data/rep100_02/traj.dcd', 'notes': 'replace_paths'},
        {'system_id': 'rep100_03', 'batch': 1, 'replicate': 3, 'length_ns': 100, 'psf_path': '/data/rep100_03/system.psf', 'dcd_path': '/data/rep100_03/traj.dcd', 'notes': 'replace_paths'},
        {'system_id': 'sys30_01', 'batch': 2, 'replicate': 1, 'length_ns': 30, 'psf_path': '/data/sys30_01/system.psf', 'dcd_path': '/data/sys30_01/traj.dcd', 'notes': 'replace_paths'},
    ])
    template.to_csv(SYSTEMS_CSV, index=False)
    raise FileNotFoundError(f'Template created: {SYSTEMS_CSV}. Edit paths and rerun.')

systems = pd.read_csv(SYSTEMS_CSV)
missing = REQUIRED_COLUMNS - set(systems.columns)
if missing:
    raise ValueError(f'systems.csv is missing required columns: {sorted(missing)}')

systems['system_id'] = systems['system_id'].astype(str)
systems['batch'] = systems['batch'].astype(int)
systems['length_ns'] = pd.to_numeric(systems['length_ns'], errors='raise')

if systems['system_id'].duplicated().any():
    dup = systems.loc[systems['system_id'].duplicated(), 'system_id'].tolist()
    raise ValueError(f'Duplicate system_id values: {dup}')

selected_systems = systems[systems['batch'].isin(RUN_BATCHES)].copy()
if selected_systems.empty:
    raise ValueError(f'No systems found for RUN_BATCHES={RUN_BATCHES}')

display(selected_systems)


In [ ]:
AA3_TO_1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','CYX':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','HID':'H','HIE':'H','HIP':'H','HSD':'H','HSE':'H','HSP':'H',
    'ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'
}
HYDROPHOBIC = {'ALA','VAL','ILE','LEU','MET','PHE','TRP','PRO'}
AROMATIC = {'PHE','TYR','TRP','HIS','HID','HIE','HIP','HSD','HSE','HSP'}
CATIONIC = {'LYS','ARG','HIS','HID','HIE','HIP','HSD','HSE','HSP'}
ACIDIC = {'ASP','GLU'}

@dataclass
class EdgeAccumulator:
    count: int = 0
    distance_sum: float = 0.0
    distance_sq_sum: float = 0.0

    def add(self, distance):
        self.count += 1
        self.distance_sum += float(distance)
        self.distance_sq_sum += float(distance) ** 2

class InteractionAccumulator:
    def __init__(self):
        self.store = defaultdict(EdgeAccumulator)

    def add(self, i, j, distance):
        if i == j:
            return
        key = (i, j) if i < j else (j, i)
        self.store[key].add(distance)

    def to_rows(self, n_frames, edge_type, min_frequency):
        rows = []
        n_frames = max(int(n_frames), 1)
        for (i, j), acc in self.store.items():
            frequency = acc.count / n_frames
            if frequency < min_frequency:
                continue
            mean_distance = acc.distance_sum / acc.count
            variance = max(acc.distance_sq_sum / acc.count - mean_distance ** 2, 0.0)
            rows.append({
                'src_idx': i,
                'dst_idx': j,
                'contactfreq': frequency,
                'meandist': mean_distance,
                'stddist': math.sqrt(variance),
                'edgetype': edge_type,
                'n_contact_frames': acc.count,
            })
        return rows

def residue_key(residue):
    return (str(residue.segid), int(residue.resid), str(residue.resname))

def safe_unit_vector(vector):
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 1e-12 else None

def angle_between_normals_deg(n1, n2):
    if n1 is None or n2 is None:
        return None
    value = np.clip(abs(float(np.dot(n1, n2))), -1.0, 1.0)
    return float(np.degrees(np.arccos(value)))

def residue_ring_selection(resid, resname, segid):
    mapping = {
        'PHE': 'name CG CD1 CD2 CE1 CE2 CZ',
        'TYR': 'name CG CD1 CD2 CE1 CE2 CZ',
        'TRP': 'name CD2 CE2 CE3 CZ2 CZ3 CH2',
        'HIS': 'name CG ND1 CD2 CE1 NE2',
        'HID': 'name CG ND1 CD2 CE1 NE2',
        'HIE': 'name CG ND1 CD2 CE1 NE2',
        'HIP': 'name CG ND1 CD2 CE1 NE2',
        'HSD': 'name CG ND1 CD2 CE1 NE2',
        'HSE': 'name CG ND1 CD2 CE1 NE2',
        'HSP': 'name CG ND1 CD2 CE1 NE2',
    }
    atom_names = mapping.get(str(resname))
    if atom_names is None:
        return 'name CA and resid -999999'
    seg_clause = f'segid {segid} and ' if segid else ''
    return f'protein and {seg_clause}resid {int(resid)} and {atom_names}'

def ring_centroid_normal(atom_group):
    if len(atom_group) < 3:
        return None, None
    positions = atom_group.positions.astype(np.float64)
    centroid = positions.mean(axis=0)
    normal = safe_unit_vector(np.cross(positions[1] - positions[0], positions[2] - positions[0]))
    return centroid, normal


In [ ]:
def build_protein_context(universe):
    protein = universe.select_atoms('protein')
    if protein.n_atoms == 0:
        raise ValueError('MDAnalysis selection `protein` returned zero atoms. Inspect PSF residue names/segments.')

    residues = list(protein.residues)
    rows = []
    resid_to_idx = {}
    for idx, residue in enumerate(residues):
        key = residue_key(residue)
        if key in resid_to_idx:
            raise ValueError(f'Duplicate protein residue identity encountered: {key}')
        resid_to_idx[key] = idx
        rows.append({
            'node_idx': idx,
            'segid': str(residue.segid),
            'resid': int(residue.resid),
            'resname': str(residue.resname),
            'aa1': AA3_TO_1.get(str(residue.resname), 'X'),
        })

    residue_table = pd.DataFrame(rows)
    atom_to_residue_idx = {atom.index: resid_to_idx[residue_key(atom.residue)] for atom in protein}

    donor = universe.select_atoms('protein and name N NE NH1 NH2 ND1 ND2 OG OG1 OH NZ NE2')
    hydrogen = universe.select_atoms('protein and name H*')
    acceptor = universe.select_atoms('protein and name O OD1 OD2 OE1 OE2 OG OG1 OH SD NE2')
    sg = universe.select_atoms('protein and resname CYS CYX and name SG')
    heavy = universe.select_atoms('protein and not name H*')

    hydrogens_by_residx = defaultdict(list)
    for atom in hydrogen:
        idx = atom_to_residue_idx.get(atom.index)
        if idx is not None:
            hydrogens_by_residx[idx].append(atom)

    residue_objects = {resid_to_idx[residue_key(r)]: r for r in residues}
    hydro_idx = [idx for idx, r in residue_objects.items() if str(r.resname) in HYDROPHOBIC]
    aromatic_idx = [idx for idx, r in residue_objects.items() if str(r.resname) in AROMATIC]
    cation_idx = [idx for idx, r in residue_objects.items() if str(r.resname) in CATIONIC]
    acid_idx = [idx for idx, r in residue_objects.items() if str(r.resname) in ACIDIC]
    basic_idx = [idx for idx, r in residue_objects.items() if str(r.resname) in CATIONIC]

    sidechain_groups = {}
    ring_groups = {}
    cation_groups = {}
    negative_groups = {}
    positive_groups = {}

    for idx, residue in residue_objects.items():
        seg_clause = f'segid {residue.segid} and ' if residue.segid else ''
        if idx in hydro_idx:
            group = universe.select_atoms(f'protein and {seg_clause}resid {int(residue.resid)} and not backbone and not name H*')
            if len(group) == 0:
                group = universe.select_atoms(f'protein and {seg_clause}resid {int(residue.resid)} and not name H*')
            sidechain_groups[idx] = group
        if idx in aromatic_idx:
            ring_groups[idx] = universe.select_atoms(residue_ring_selection(residue.resid, residue.resname, residue.segid))
        if idx in cation_idx:
            cation_groups[idx] = universe.select_atoms(f'protein and {seg_clause}resid {int(residue.resid)} and name NZ NH1 NH2 NE ND1 NE2')
        if idx in acid_idx:
            negative_groups[idx] = universe.select_atoms(f'protein and {seg_clause}resid {int(residue.resid)} and name OD1 OD2 OE1 OE2')
        if idx in basic_idx:
            positive_groups[idx] = universe.select_atoms(f'protein and {seg_clause}resid {int(residue.resid)} and name NZ NH1 NH2 NE ND1 NE2')

    ca_by_idx = {}
    for idx, residue in residue_objects.items():
        seg_clause = f'segid {residue.segid} and ' if residue.segid else ''
        ca = universe.select_atoms(f'protein and {seg_clause}resid {int(residue.resid)} and name CA')
        if len(ca):
            ca_by_idx[idx] = ca

    return {
        'protein': protein,
        'residue_table': residue_table,
        'atom_to_residue_idx': atom_to_residue_idx,
        'hydrogens_by_residx': hydrogens_by_residx,
        'donor': donor,
        'acceptor': acceptor,
        'sg': sg,
        'heavy': heavy,
        'hydro_idx': hydro_idx,
        'aromatic_idx': aromatic_idx,
        'cation_idx': cation_idx,
        'acid_idx': acid_idx,
        'basic_idx': basic_idx,
        'sidechain_groups': sidechain_groups,
        'ring_groups': ring_groups,
        'cation_groups': cation_groups,
        'negative_groups': negative_groups,
        'positive_groups': positive_groups,
        'ca_by_idx': ca_by_idx,
    }

def valid_nonsequential(i, j):
    return abs(int(i) - int(j)) >= SEQUENCE_SEPARATION_MIN

def update_hbond(acc, ctx):
    donor, acceptor = ctx['donor'], ctx['acceptor']
    if len(donor) == 0 or len(acceptor) == 0:
        return
    pairs, distances = capped_distance(donor.positions, acceptor.positions, max_cutoff=HBOND_DISTANCE_A, return_distances=True)
    amap = ctx['atom_to_residue_idx']
    hydrogens = ctx['hydrogens_by_residx']
    for (i, j), distance in zip(pairs, distances):
        donor_atom, acceptor_atom = donor[int(i)], acceptor[int(j)]
        ri = amap.get(donor_atom.index)
        rj = amap.get(acceptor_atom.index)
        if ri is None or rj is None or ri == rj or not valid_nonsequential(ri, rj):
            continue
        for hydrogen_atom in hydrogens.get(ri, []):
            try:
                angle = float(np.degrees(calc_angles(donor_atom.position, hydrogen_atom.position, acceptor_atom.position)))
                if np.isfinite(angle) and angle >= HBOND_ANGLE_DEG:
                    acc.add(ri, rj, distance)
                    break
            except Exception:
                continue

def update_disulfide(acc, ctx):
    sg = ctx['sg']
    if len(sg) < 2:
        return
    pairs, distances = capped_distance(sg.positions, sg.positions, max_cutoff=2.2, return_distances=True)
    amap = ctx['atom_to_residue_idx']
    for (i, j), distance in zip(pairs, distances):
        if int(i) >= int(j):
            continue
        ri, rj = amap.get(sg[int(i)].index), amap.get(sg[int(j)].index)
        if ri is not None and rj is not None and ri != rj:
            acc.add(ri, rj, distance)

def update_vdw(acc, ctx):
    heavy = ctx['heavy']
    if len(heavy) == 0:
        return
    pairs, distances = capped_distance(heavy.positions, heavy.positions, max_cutoff=VDW_MAX_A, return_distances=True)
    amap = ctx['atom_to_residue_idx']
    for (i, j), distance in zip(pairs, distances):
        if int(i) >= int(j) or distance <= VDW_MIN_A:
            continue
        ri, rj = amap.get(heavy[int(i)].index), amap.get(heavy[int(j)].index)
        if ri is not None and rj is not None and ri != rj and valid_nonsequential(ri, rj):
            acc.add(ri, rj, distance)

def update_hydrophobic(acc, ctx):
    coords, indices = [], []
    for idx in ctx['hydro_idx']:
        group = ctx['sidechain_groups'].get(idx)
        if group is not None and len(group):
            coords.append(group.center_of_mass())
            indices.append(idx)
    if len(coords) < 2:
        return
    pairs, distances = capped_distance(np.asarray(coords), np.asarray(coords), max_cutoff=HYDROPHOBIC_CUTOFF_A, return_distances=True)
    for (i, j), distance in zip(pairs, distances):
        if int(i) >= int(j):
            continue
        ri, rj = indices[int(i)], indices[int(j)]
        if valid_nonsequential(ri, rj):
            acc.add(ri, rj, distance)

def get_ring_data(ctx):
    output = []
    for idx in ctx['aromatic_idx']:
        group = ctx['ring_groups'].get(idx)
        if group is None:
            continue
        centroid, normal = ring_centroid_normal(group)
        if centroid is not None and normal is not None:
            output.append((idx, centroid, normal))
    return output

def update_aromatic_pi(acc, ctx):
    ring_data = get_ring_data(ctx)
    if len(ring_data) < 2:
        return
    coords = np.asarray([x[1] for x in ring_data])
    pairs, distances = capped_distance(coords, coords, max_cutoff=AROMATIC_PI_CUTOFF_A, return_distances=True)
    for (i, j), distance in zip(pairs, distances):
        if int(i) >= int(j):
            continue
        ri, _, ni = ring_data[int(i)]
        rj, _, nj = ring_data[int(j)]
        if not valid_nonsequential(ri, rj):
            continue
        angle = angle_between_normals_deg(ni, nj)
        if angle is not None and (angle <= AROMATIC_PARALLEL_MAX_DEG or AROMATIC_T_MIN_DEG <= angle <= AROMATIC_T_MAX_DEG):
            acc.add(ri, rj, distance)

def update_cation_pi(acc, ctx):
    cations = []
    for idx in ctx['cation_idx']:
        group = ctx['cation_groups'].get(idx)
        if group is not None and len(group):
            cations.append((idx, group.center_of_mass()))
    rings = get_ring_data(ctx)
    if not cations or not rings:
        return
    cat_coords = np.asarray([x[1] for x in cations])
    ring_coords = np.asarray([x[1] for x in rings])
    pairs, distances = capped_distance(cat_coords, ring_coords, max_cutoff=CATION_PI_CUTOFF_A, return_distances=True)
    for (i, j), distance in zip(pairs, distances):
        ri, cation_coord = cations[int(i)]
        rj, ring_centroid, ring_normal = rings[int(j)]
        if ri == rj or not valid_nonsequential(ri, rj):
            continue
        vector = safe_unit_vector(cation_coord - ring_centroid)
        if vector is None:
            continue
        angle = float(np.degrees(np.arccos(np.clip(abs(float(np.dot(vector, ring_normal))), -1.0, 1.0))))
        if angle <= CATION_PI_NORMAL_MAX_DEG:
            acc.add(ri, rj, distance)

def update_ionic(acc, ctx, cutoff):
    acidic, basic = [], []
    for idx in ctx['acid_idx']:
        group = ctx['negative_groups'].get(idx)
        if group is not None and len(group):
            acidic.append((idx, group.center_of_mass()))
    for idx in ctx['basic_idx']:
        group = ctx['positive_groups'].get(idx)
        if group is not None and len(group):
            basic.append((idx, group.center_of_mass()))
    if not acidic or not basic:
        return
    acid_coords = np.asarray([x[1] for x in acidic])
    basic_coords = np.asarray([x[1] for x in basic])
    pairs, distances = capped_distance(acid_coords, basic_coords, max_cutoff=cutoff, return_distances=True)
    for (i, j), distance in zip(pairs, distances):
        ri, rj = acidic[int(i)][0], basic[int(j)][0]
        if ri != rj and valid_nonsequential(ri, rj):
            acc.add(ri, rj, distance)

def add_backbone_edges(ctx):
    rows = []
    table = ctx['residue_table']
    ca_by_idx = ctx['ca_by_idx']
    for idx in range(len(table) - 1):
        left, right = table.iloc[idx], table.iloc[idx + 1]
        if str(left['segid']) != str(right['segid']):
            continue
        if int(right['resid']) - int(left['resid']) != 1:
            continue
        ca_left, ca_right = ca_by_idx.get(idx), ca_by_idx.get(idx + 1)
        if ca_left is None or ca_right is None:
            continue
        distance = float(np.linalg.norm(ca_left.positions[0] - ca_right.positions[0]))
        if distance <= BACKBONE_CA_MAX_A:
            rows.append({
                'src_idx': idx, 'dst_idx': idx + 1, 'contactfreq': 1.0,
                'meandist': distance, 'stddist': 0.0, 'edgetype': 'backbone', 'n_contact_frames': 0
            })
    return rows


In [ ]:
def run_pbc_diagnostic(universe, ctx, system_id):
    n_frames = len(universe.trajectory)
    candidate_frames = sorted(set([0, max(0, n_frames // 2), max(0, n_frames - 1)]))
    records = []
    ca = ctx['protein'].select_atoms('name CA')
    for frame_index in candidate_frames:
        universe.trajectory[frame_index]
        span = np.ptp(ca.positions, axis=0) if len(ca) else np.array([np.nan, np.nan, np.nan])
        dimensions = universe.trajectory.ts.dimensions
        records.append({
            'system_id': system_id,
            'frame': int(frame_index),
            'n_ca': int(len(ca)),
            'span_x_A': float(span[0]),
            'span_y_A': float(span[1]),
            'span_z_A': float(span[2]),
            'box_x_A': float(dimensions[0]) if dimensions is not None and len(dimensions) >= 3 else np.nan,
            'box_y_A': float(dimensions[1]) if dimensions is not None and len(dimensions) >= 3 else np.nan,
            'box_z_A': float(dimensions[2]) if dimensions is not None and len(dimensions) >= 3 else np.nan,
        })
    return pd.DataFrame(records)

def maybe_write_protein_dcd(universe, ctx, output_path):
    if output_path.exists():
        return output_path
    protein = ctx['protein']
    with mda.Writer(str(output_path), n_atoms=protein.n_atoms) as writer:
        for _ in tqdm(universe.trajectory, total=len(universe.trajectory), desc=f'Cache {output_path.stem}', leave=False):
            writer.write(protein)
    return output_path

def build_consensus_edges(universe, ctx, system_id):
    accumulators = {
        'hbond': InteractionAccumulator(),
        'disulfide': InteractionAccumulator(),
        'vdw': InteractionAccumulator(),
        'hydrophobic': InteractionAccumulator(),
        'aromaticpi': InteractionAccumulator(),
        'cationpi': InteractionAccumulator(),
        'ionic': InteractionAccumulator(),
        'saltbridge': InteractionAccumulator(),
    }

    frame_indices = list(range(0, len(universe.trajectory), max(1, FRAME_STRIDE)))
    for frame_index in tqdm(frame_indices, desc=f'Contacts {system_id}', leave=False):
        universe.trajectory[frame_index]
        update_hbond(accumulators['hbond'], ctx)
        update_disulfide(accumulators['disulfide'], ctx)
        update_vdw(accumulators['vdw'], ctx)
        update_hydrophobic(accumulators['hydrophobic'], ctx)
        update_aromatic_pi(accumulators['aromaticpi'], ctx)
        update_cation_pi(accumulators['cationpi'], ctx)
        update_ionic(accumulators['ionic'], ctx, IONIC_CUTOFF_A)
        update_ionic(accumulators['saltbridge'], ctx, SALTBRIDGE_CUTOFF_A)

    rows = add_backbone_edges(ctx)
    for edge_type, accumulator in accumulators.items():
        rows.extend(accumulator.to_rows(len(frame_indices), edge_type, MIN_CONTACT_FREQUENCY))

    edge_table = pd.DataFrame(rows)
    if edge_table.empty:
        return edge_table, len(frame_indices)

    metadata = ctx['residue_table'].set_index('node_idx')
    edge_table['src'] = edge_table['src_idx'].map(metadata['resid'])
    edge_table['dst'] = edge_table['dst_idx'].map(metadata['resid'])
    edge_table['src_resname'] = edge_table['src_idx'].map(metadata['resname'])
    edge_table['dst_resname'] = edge_table['dst_idx'].map(metadata['resname'])
    edge_table['src_segid'] = edge_table['src_idx'].map(metadata['segid'])
    edge_table['dst_segid'] = edge_table['dst_idx'].map(metadata['segid'])
    edge_table = edge_table.sort_values(['edgetype', 'src_idx', 'dst_idx']).reset_index(drop=True)
    return edge_table, len(frame_indices)

def build_graph(residue_table, edge_table):
    graph = nx.Graph()
    for row in residue_table.itertuples(index=False):
        graph.add_node(int(row.node_idx), resid=int(row.resid), resname=str(row.resname), segid=str(row.segid), aa1=str(row.aa1))

    # A simple Graph can store only one edge per residue pair. When contact types overlap,
    # retain the largest occupancy edge and record all observed types.
    for row in edge_table.itertuples(index=False):
        i, j = int(row.src_idx), int(row.dst_idx)
        data = {
            'contactfreq': float(row.contactfreq),
            'meandist': float(row.meandist),
            'stddist': float(row.stddist),
            'edgetype': str(row.edgetype),
            'edge_types': str(row.edgetype),
        }
        if graph.has_edge(i, j):
            existing = graph.edges[i, j]
            existing_types = set(str(existing.get('edge_types', existing.get('edgetype', ''))).split(';'))
            existing_types.add(str(row.edgetype))
            existing['edge_types'] = ';'.join(sorted(x for x in existing_types if x))
            if float(row.contactfreq) > float(existing.get('contactfreq', 0.0)):
                existing.update(data)
                existing['edge_types'] = ';'.join(sorted(existing_types))
        else:
            graph.add_edge(i, j, **data)
    return graph

def graph_qc(graph, edge_table):
    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()
    components = sorted(nx.connected_components(graph), key=len, reverse=True) if n_nodes else []
    giant_size = len(components[0]) if components else 0
    type_counts = edge_table['edgetype'].value_counts().to_dict() if len(edge_table) else {}
    return {
        'n_nodes': int(n_nodes),
        'n_edges': int(n_edges),
        'density': float(nx.density(graph)) if n_nodes > 1 else np.nan,
        'n_components': int(len(components)),
        'giant_component_size': int(giant_size),
        'giant_component_fraction': float(giant_size / n_nodes) if n_nodes else np.nan,
        'backbone_edges': int(type_counts.get('backbone', 0)),
        'edge_types_present': ';'.join(sorted(type_counts)),
        'edge_type_counts_json': json.dumps(type_counts, sort_keys=True),
    }


In [ ]:
def compute_centralities(graph, residue_table):
    output = residue_table.copy()
    n_nodes = graph.number_of_nodes()
    if n_nodes == 0:
        for column in ['degree', 'strength', 'betweenness', 'closeness', 'eigenvector', 'pagerank', 'kcore', 'component_id', 'in_giant_component']:
            output[column] = np.nan
        return output

    components = sorted(nx.connected_components(graph), key=len, reverse=True)
    giant = components[0] if components else set()
    component_map = {}
    for component_id, members in enumerate(components):
        for node in members:
            component_map[node] = component_id

    degree = dict(graph.degree())
    strength = {node: float(sum(data.get('contactfreq', 0.0) for _, _, data in graph.edges(node, data=True))) for node in graph.nodes()}
    kcore = nx.core_number(graph) if graph.number_of_edges() else {node: 0 for node in graph.nodes()}

    target = graph.subgraph(giant).copy() if CENTRALITY_ON_GIANT_COMPONENT and giant else graph.copy()
    for _, _, data in target.edges(data=True):
        data['distance_weight'] = 1.0 / max(float(data.get('contactfreq', 0.0)), 1e-9)

    if target.number_of_nodes() > 1 and target.number_of_edges() > 0:
        betweenness_target = nx.betweenness_centrality(target, weight='distance_weight', normalized=True)
        closeness_target = nx.closeness_centrality(target, distance='distance_weight')
        try:
            eigenvector_target = nx.eigenvector_centrality_numpy(target, weight='contactfreq')
        except Exception:
            eigenvector_target = nx.eigenvector_centrality(target, weight='contactfreq', max_iter=5000, tol=1e-8)
        pagerank_target = nx.pagerank(target, weight='contactfreq')
    else:
        betweenness_target = {node: 0.0 for node in target.nodes()}
        closeness_target = {node: 0.0 for node in target.nodes()}
        eigenvector_target = {node: 0.0 for node in target.nodes()}
        pagerank_target = {node: 0.0 for node in target.nodes()}

    output['degree'] = output['node_idx'].map(degree).fillna(0).astype(int)
    output['strength'] = output['node_idx'].map(strength).fillna(0.0)
    output['betweenness'] = output['node_idx'].map(betweenness_target).fillna(0.0)
    output['closeness'] = output['node_idx'].map(closeness_target).fillna(0.0)
    output['eigenvector'] = output['node_idx'].map(eigenvector_target).fillna(0.0)
    output['pagerank'] = output['node_idx'].map(pagerank_target).fillna(0.0)
    output['kcore'] = output['node_idx'].map(kcore).fillna(0).astype(int)
    output['component_id'] = output['node_idx'].map(component_map).fillna(-1).astype(int)
    output['in_giant_component'] = output['node_idx'].isin(giant)

    output['mean_incident_contactfreq'] = np.divide(
        output['strength'], output['degree'],
        out=np.zeros(len(output), dtype=float), where=output['degree'].to_numpy() > 0
    )

    def hub_mask(metric):
        values = output.loc[output['in_giant_component'], metric]
        if values.empty:
            return pd.Series(False, index=output.index)
        threshold = float(values.quantile(HUB_QUANTILE))
        mask = (output[metric] >= threshold) & output['in_giant_component']
        if int(mask.sum()) < MIN_HUBS_IF_NONEMPTY and len(values) >= MIN_HUBS_IF_NONEMPTY:
            top_indices = output.loc[output['in_giant_component']].nlargest(MIN_HUBS_IF_NONEMPTY, metric).index
            mask.loc[top_indices] = True
        return mask

    output['is_hub_degree'] = hub_mask('degree')
    output['is_hub_betweenness'] = hub_mask('betweenness')
    output['is_hub_consensus'] = output['is_hub_degree'] & output['is_hub_betweenness']
    output['degree_rank_desc'] = output['degree'].rank(method='min', ascending=False).astype(int)
    output['betweenness_rank_desc'] = output['betweenness'].rank(method='min', ascending=False).astype(int)
    output['hub_score'] = output[['is_hub_degree', 'is_hub_betweenness']].sum(axis=1).astype(int)
    return output

def update_status(status_rows):
    status = pd.DataFrame(status_rows)
    status.to_csv(RESULTS_DIR / 'status.csv', index=False)
    return status

def process_system(system_row):
    system_id = str(system_row['system_id'])
    started = time.time()
    psf_path = Path(str(system_row['psf_path']))
    dcd_path = Path(str(system_row['dcd_path']))

    if not psf_path.exists():
        raise FileNotFoundError(f'PSF not found: {psf_path}')
    if not dcd_path.exists():
        raise FileNotFoundError(f'DCD not found: {dcd_path}')

    universe = mda.Universe(str(psf_path), str(dcd_path))
    ctx = build_protein_context(universe)

    pbc = run_pbc_diagnostic(universe, ctx, system_id)
    pbc.to_csv(PER_SYSTEM_DIR / f'{system_id}_pbc_diagnostic.csv', index=False)

    if WRITE_PROTEIN_ONLY_DCD_CACHE:
        cache_path = CACHE_DIR / f'{system_id}_protein.dcd'
        maybe_write_protein_dcd(universe, ctx, cache_path)

    edge_table, n_frames_used = build_consensus_edges(universe, ctx, system_id)
    graph = build_graph(ctx['residue_table'], edge_table)
    qc = graph_qc(graph, edge_table)
    centrality = compute_centralities(graph, ctx['residue_table'])

    metadata = system_row.to_dict()
    for key, value in metadata.items():
        centrality[key] = value
    centrality['n_frames_total'] = len(universe.trajectory)
    centrality['n_frames_used'] = n_frames_used
    centrality['frame_stride'] = FRAME_STRIDE
    centrality['min_contact_frequency'] = MIN_CONTACT_FREQUENCY

    edge_table_out = edge_table.copy()
    for key, value in metadata.items():
        edge_table_out[key] = value
    edge_table_out['n_frames_total'] = len(universe.trajectory)
    edge_table_out['n_frames_used'] = n_frames_used

    hubs = centrality[centrality['is_hub_consensus']].copy()
    hubs = hubs.sort_values(['betweenness', 'degree'], ascending=False)

    centrality.to_csv(PER_SYSTEM_DIR / f'{system_id}_centrality.csv', index=False)
    edge_table_out.to_csv(PER_SYSTEM_DIR / f'{system_id}_edges.csv', index=False)
    hubs.to_csv(PER_SYSTEM_DIR / f'{system_id}_hubs.csv', index=False)

    summary = {
        **metadata,
        **qc,
        'n_frames_total': int(len(universe.trajectory)),
        'n_frames_used': int(n_frames_used),
        'frame_stride': int(FRAME_STRIDE),
        'min_contact_frequency': float(MIN_CONTACT_FREQUENCY),
        'n_degree_hubs': int(centrality['is_hub_degree'].sum()),
        'n_betweenness_hubs': int(centrality['is_hub_betweenness'].sum()),
        'n_consensus_hubs': int(centrality['is_hub_consensus'].sum()),
        'mean_degree': float(centrality['degree'].mean()),
        'mean_strength': float(centrality['strength'].mean()),
        'runtime_min': round((time.time() - started) / 60, 3),
        'status': 'done',
    }
    (PER_SYSTEM_DIR / f'{system_id}_qc.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

    del universe, ctx, graph, edge_table, centrality
    gc.collect()
    return summary


In [ ]:
STATUS_PATH = RESULTS_DIR / 'status.csv'
if STATUS_PATH.exists():
    status_history = pd.read_csv(STATUS_PATH)
else:
    status_history = pd.DataFrame()

completed = set()
if len(status_history) and 'status' in status_history.columns:
    completed = set(status_history.loc[status_history['status'].eq('done'), 'system_id'].astype(str))

status_rows = []
if len(status_history):
    # Keep only the latest status per system for a clean incremental state file.
    status_rows = status_history.drop_duplicates('system_id', keep='last').to_dict('records')

for _, system_row in selected_systems.iterrows():
    system_id = str(system_row['system_id'])
    if system_id in completed and not FORCE_RERUN:
        print(f'SKIP {system_id}: already done')
        continue

    print(f'\n=== START {system_id} ===')
    status_rows = [r for r in status_rows if str(r.get('system_id')) != system_id]
    try:
        result = process_system(system_row)
        status_rows.append(result)
        print(f"DONE {system_id}: {result['n_edges']} graph edges; {result['runtime_min']:.2f} min")
    except Exception as exc:
        failure = {
            **system_row.to_dict(),
            'status': 'failed',
            'error': repr(exc),
            'traceback': traceback.format_exc(),
            'runtime_min': np.nan,
        }
        status_rows.append(failure)
        print(f'FAILED {system_id}: {exc}')
    finally:
        update_status(status_rows)
        gc.collect()

status = pd.read_csv(STATUS_PATH)
display(status)


In [ ]:
# Aggregate all successfully generated per-system tables. This cell can be rerun after batch 1 and batch 2.
centrality_files = sorted(PER_SYSTEM_DIR.glob('*_centrality.csv'))
edge_files = sorted(PER_SYSTEM_DIR.glob('*_edges.csv'))
hub_files = sorted(PER_SYSTEM_DIR.glob('*_hubs.csv'))

if not centrality_files:
    raise FileNotFoundError('No per-system centrality files found. Run the processing cell first.')

summary_all = pd.concat([pd.read_csv(path) for path in centrality_files], ignore_index=True)
summary_all.to_csv(RESULTS_DIR / 'summary_all_systems.csv', index=False)

summary_all_hubs = pd.concat([pd.read_csv(path) for path in hub_files], ignore_index=True) if hub_files else pd.DataFrame()
summary_all_hubs.to_csv(RESULTS_DIR / 'summary_all_hubs.csv', index=False)

summary_all_edges = pd.concat([pd.read_csv(path) for path in edge_files], ignore_index=True) if edge_files else pd.DataFrame()
summary_all_edges.to_csv(RESULTS_DIR / 'summary_all_edges.csv', index=False)

status = pd.read_csv(STATUS_PATH) if STATUS_PATH.exists() else pd.DataFrame()
status.to_csv(RESULTS_DIR / 'summary_system_qc.csv', index=False)

failed = status[status['status'].eq('failed')].copy() if len(status) and 'status' in status.columns else pd.DataFrame()
failed.to_csv(RESULTS_DIR / 'failed_systems.csv', index=False)

print('Saved:')
print(' - summary_all_systems.csv:', len(summary_all), 'residue rows')
print(' - summary_all_hubs.csv:', len(summary_all_hubs), 'hub rows')
print(' - summary_all_edges.csv:', len(summary_all_edges), 'edge rows')
print(' - summary_system_qc.csv:', len(status), 'system rows')
print(' - failed_systems.csv:', len(failed), 'failed systems')


In [ ]:
# Cross-trajectory hub recurrence is descriptive only: no pooling of trajectories into one consensus graph.
hub_input = summary_all.copy()
present = (
    hub_input.groupby(['segid', 'resid', 'resname'], dropna=False)['system_id']
    .nunique()
    .rename('n_systems_present')
    .reset_index()
)
recurrence = (
    hub_input[hub_input['is_hub_consensus'].astype(bool)]
    .groupby(['segid', 'resid', 'resname'], dropna=False)['system_id']
    .nunique()
    .rename('n_consensus_hub')
    .reset_index()
)
hub_recurrence = present.merge(recurrence, on=['segid', 'resid', 'resname'], how='left')
hub_recurrence['n_consensus_hub'] = hub_recurrence['n_consensus_hub'].fillna(0).astype(int)
hub_recurrence['hub_recurrence'] = hub_recurrence['n_consensus_hub'] / hub_recurrence['n_systems_present']
hub_recurrence = hub_recurrence.sort_values(['hub_recurrence', 'n_consensus_hub', 'resid'], ascending=[False, False, True])
hub_recurrence.to_csv(RESULTS_DIR / 'hub_recurrence.csv', index=False)

display(hub_recurrence.head(30))


## Interpretation and QC notes

- **Do not merge trajectories into a single graph.** Each PSF/DCD receives its own consensus RIN; `hub_recurrence.csv` only reports how often a residue satisfies the hub definition across independent trajectories.
- Before interpreting hubs, inspect `*_pbc_diagnostic.csv`. If a protein crosses a periodic boundary and appears fragmented, coordinates must be made whole before graph construction. This notebook deliberately does not apply automatic PBC transformations because DCD files may lack complete unit-cell information and automatic repairs can silently alter coordinates.
- `backbone` edges are included to maintain sequential chain connectivity. Non-backbone interaction types exclude close sequential neighbours (`SEQUENCE_SEPARATION_MIN=2`) to reduce trivial covalent-neighbour contacts.
- In a simple `nx.Graph`, an amino-acid pair with several contact classes remains one graph edge. Its `contactfreq` is the maximum occupancy among the retained contact classes, and `edge_types` contains the full semicolon-separated set.
- The 6-hour target is plausible only if PSF/DCD files are locally accessible and frame count/atom count are moderate. The main cost is evaluating contact types over sampled frames, not NetworkX centrality. Run one representative 100-ns trajectory first, inspect `runtime_min` in `summary_system_qc.csv`, then estimate the complete wall time.
